In [1]:
# Import packages
import numpy as np
import pandas as pd
import anndata
import scanpy as sc
import matplotlib.pyplot as plt
import re
import os
import sys
from scipy.sparse import csr_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
import matplotlib
import harmony

findfont: Font family ['Raleway'] not found. Falling back to DejaVu Sans.
findfont: Font family ['Lato'] not found. Falling back to DejaVu Sans.


In [2]:
adata_patient = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/adatas/KG146_Patient_Organoid.h5ad')

In [9]:
adata_patient.obs['Cell State'].value_counts()

Cell State
NA                  9784
nan                  928
SCC                  280
ISC-like             252
Injury Repair        230
Enterocyte-like      223
Fetal Progenitor     106
TA-like               80
Goblet-like           71
Early NET             62
Name: count, dtype: int64

In [10]:
adata_patient = adata_patient[~adata_patient.obs.loc[:,'Cell State'].isin(['nan','NA']),:].copy()

In [11]:
adata_patient.obs['Cell State'].value_counts()

Cell State
SCC                 280
ISC-like            252
Injury Repair       230
Enterocyte-like     223
Fetal Progenitor    106
TA-like              80
Goblet-like          71
Early NET            62
Name: count, dtype: int64

In [17]:
probabilities = pd.read_csv('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_noreplicates/harmony_phenograph/061124/adata.probabilities.Metastatic_BASE_CTRL.csv')

In [18]:
probabilities

,Unnamed: 0,Early NET,Enterocyte-like,Fetal Progenitor,Goblet-like,ISC-like,Injury Repair,SCC,TA-like,Cell_State
0,KG146Li_BASE_shCtrl_GCCAGCATCAGACCGC-1,0.000699,0.147305,0.000457,0.050580,0.100395,0.010906,0.000062,0.689596,TA-like
1,KG146Li_BASE_shCtrl_GTGAGCCCATGACTAC-1,0.001369,0.189278,0.000958,0.049763,0.154335,0.021178,0.000189,0.582930,TA-like
2,KG146Li_BASE_shCtrl_GTGAGGAGTATCCTCC-1,0.005038,0.173851,0.004238,0.058505,0.255316,0.069418,0.001200,0.432433,TA-like
3,KG146Li_BASE_shCtrl_AACCATGTCAGAGTTC-1,0.000232,0.060817,0.000170,0.020377,0.032485,0.003230,0.000019,0.882669,TA-like
4,KG146Li_BASE_shCtrl_TGGCGTGAGCATTTGC-1,0.000638,0.149925,0.000391,0.037230,0.111303,0.010772,0.000056,0.689685,TA-like
...,...,...,...,...,...,...,...,...,...,...
2181,KG146Li_BASE_shCtrl_AGCGATTGTAGCTGCC-1,0.000695,0.760626,0.000199,0.008724,0.133229,0.072585,0.000046,0.023896,Enterocyte-like
2182,KG146Li_BASE_shCtrl_GTAGCTAAGGGACAGG-1,0.025573,0.332036,0.014017,0.423375,0.139878,0.056892,0.000021,0.008208,Goblet-like
2183,KG146Li_BASE_shCtrl_CATAAGCTCAGTGATC-1,0.060343,0.281081,0.127817,0.243664,0.091688,0.181560,0.000050,0.013796,Enterocyte-like
2184,KG146Li_BASE_shCtrl_CGTGCTTCAATTCTTC-1,0.009774,0.235702,0.004167,0.678467,0.032920,0.035614,0.000008,0.003349,Goblet-like


In [8]:
probabilities['Cell_State'].value_counts()

Cell_State
TA-like             809
Enterocyte-like     741
Goblet-like         296
ISC-like            247
Injury Repair        62
Early NET            25
Fetal Progenitor      6
Name: count, dtype: int64

In [2]:
#adata_org = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output.042224/adata.organoid.combined.042224.h5ad')
adata_patient = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/adatas/KG146_Patient_Organoid.h5ad')
adata_org=sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_noreplicates/stratify_with_zfpkd/Metastatic_BASE_CTRL.h5ad')

In [3]:
# Functions 
'''CONSTRUCT AFFINITY MATRIX'''
def _convert_to_affinity(adj, scaling_factors, device, with_self_loops=False):
    """ Convert adjacency matrix to affinity matrix
    """
    N = adj.shape[0]
    rows, cols, dists = find(adj)
    if device == "gpu":
        import cupy as cp
        from cupyx.scipy.sparse import csr_matrix as csr_matrix_gpu
        dists = cp.array(dists) ** 2/(cp.array(scaling_factors.values[rows]) ** 2)
        rows, cols = cp.array(rows), cp.array(cols)
        # Self loops
        if with_self_loops:
            dists = cp.append(dists, cp.zeros(N))
            rows = cp.append(rows, range(N))
            cols = cp.append(cols, range(N))
        aff = csr_matrix_gpu((cp.exp(-dists), (rows, cols)), shape=(N, N)).get()
    elif device == "cpu":
        dists = dists ** 2/(scaling_factors.values[rows] ** 2)

        # Self loops
        if with_self_loops:
            dists = np.append(dists, np.zeros(N))
            rows = np.append(rows, range(N))
            cols = np.append(cols, range(N))
        aff = csr_matrix((np.exp(-dists), (rows, cols)), shape=[N, N])
    return aff

'''CONSTRUCT MUTUAL NEAREST NEIGHBORS GRAPH'''
def _construct_mnn(t1_cells, t2_cells, data_df, n_neighbors,device,n_jobs=-2):
    # FUnction to construct mutually nearest neighbors bewteen two points
    
    if device == "gpu":
        from cuml import NearestNeighbors
        nbrs = NearestNeighbors(n_neighbors=n_neighbors,
                                metric='cosine')
    elif device == "cpu":
        from sklearn.neighbors import NearestNeighbors
        nbrs = NearestNeighbors(n_neighbors=n_neighbors,
                                metric='cosine', n_jobs=n_jobs)
    
    print(f't+1 neighbors of t...')
    nbrs.fit(data_df.loc[t1_cells, :].values)
    t1_nbrs = nbrs.kneighbors_graph(
        data_df.loc[t2_cells, :].values, mode='distance')

    print(f't neighbors of t+1...')
    nbrs.fit(data_df.loc[t2_cells, :].values)
    t2_nbrs = nbrs.kneighbors_graph(
        data_df.loc[t1_cells, :].values, mode='distance')

    # Mututally nearest neighbors
    mnn = t2_nbrs.multiply(t1_nbrs.T)
    mnn = mnn.sqrt()
    return mnn

'''COMPUTE SCALING FACTORS'''
def _mnn_scaling_factors(mnn_ka_dists, scaling_factors,device):
    if device == "gpu":
        from cuml import LinearRegression
    else:
        from sklearn.linear_model import LinearRegression
    cells = mnn_ka_dists.index[~mnn_ka_dists.isnull()]

    # Linear model fit
    x = scaling_factors[cells]
    y = mnn_ka_dists[cells]
    lm = LinearRegression()
    lm.fit(x.values.reshape(-1, 1), y.values.reshape(-1, 1))

    # Predict
    x = scaling_factors[mnn_ka_dists.index]
    vals = np.ravel(lm.predict(x.values.reshape(-1, 1)))
    mnn_scaling_factors = pd.Series(vals, index=mnn_ka_dists.index)

    return mnn_scaling_factors

'''CONSTRUCT AFFINITY MATRIX'''
def _mnn_affinity(mnn, mnn_scaling_factors, offset1, offset2, device):
    # Function to convert mnn matrix to affinicty matrix

    # Construct adjacency matrix
    N = len(mnn_scaling_factors)
    x, y, z = find(mnn)
    x = x + offset1
    y = y + offset2
    adj = csr_matrix((z, (x, y)), shape=[N, N])

    # Affinity matrix
    return _convert_to_affinity(adj, mnn_scaling_factors, device, False)

'''
HARMONY AND PALANTIR
'''
def _mnn_ka_distances(mnn, n_neighbors):
    # Function to find distance ka^th neighbor in the mutual nearest neighbor matrix
    ka = int(n_neighbors / 3)
    ka_dists = np.repeat(None, mnn.shape[0])
    x, y, z = find(mnn)
    rows=pd.Series(x).value_counts()
    for r in rows.index[rows >= ka]:
        ka_dists[r] = np.sort(z[x==r])[ka - 1]
    return ka_dists

from harmony import core
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import find, csr_matrix

def harmony_aug_mat_with_pca(projections, timepoints, timepoint_connections, n_neighbors, n_neighbors2):
    # Time point cells and indices
    tp_cells = pd.Series()
    tp_offset = pd.Series()
    offset = 0
    for i in timepoints.unique():
        tp_offset[i] = offset
        tp_cells[i] = list(timepoints.index[timepoints == i])
        offset += len(tp_cells[i])

    # Nearest neighbor graph construction and affinity matrix
    print('Nearest neighbor computation...')
    nbrs = NearestNeighbors(n_neighbors=n_neighbors,
                            metric='cosine', n_jobs=-2).fit(projections.values)

    adj = nbrs.kneighbors_graph(projections.values, mode='distance')
    dists, _ = nbrs.kneighbors(projections.values)
    
    # Scaling factors for affinity matrix construction
    ka = int(n_neighbors / 3)
    scaling_factors = pd.Series(dists[:, ka], index=projections.index)
    
    # Affinity matrix
    nn_aff = _convert_to_affinity(adj, scaling_factors, 'cpu', True)
    n_jobs = -2
    
    # Mututally nearest neighbor affinity matrix
    # Initilze mnn affinity matrix
    N = projections.shape[0]
    full_mnn_aff = csr_matrix(([0], ([0], [0])), [N, N])
    for i in timepoint_connections.index:
        t1, t2 = timepoint_connections.loc[i, :].values
        print(f'Constucting affinities between {t1} and {t2}...')
        # MNN matrix  and distance to ka the distance
        t1_cells = tp_cells[t1]
        t2_cells = tp_cells[t2]
        mnn = _construct_mnn(t1_cells, t2_cells, projections,
                             n_neighbors2, 'cpu', n_jobs)
        
        # MNN Scaling factors
        # Distance to the adaptive neighbor
        ka_dists = pd.Series(0.0, index=t1_cells + t2_cells)
        ka_dists = ka_dists.astype(float)
        # T1 scaling factors
        ka_dists[t1_cells] = _mnn_ka_distances(mnn, n_neighbors2)
        # T2 scaling factors
        ka_dists[t2_cells] = _mnn_ka_distances(mnn.T, n_neighbors2)
        # Scaling factors
        mnn_scaling_factors = pd.Series(0.0, index=projections.index)
#         mnn_scaling_factors[t1_cells] = core._mnn_scaling_factors(
#             ka_dists[t1_cells], scaling_factors,'cpu')
#         mnn_scaling_factors[t2_cells] = core._mnn_scaling_factors(
#             ka_dists[t2_cells], scaling_factors,'cpu')
        mnn_scaling_factors[t1_cells] = _mnn_scaling_factors(
            ka_dists[t1_cells], scaling_factors,'cpu')
        mnn_scaling_factors[t2_cells] = _mnn_scaling_factors(
            ka_dists[t2_cells], scaling_factors,'cpu')
        
        # MNN affinity matrix
#         full_mnn_aff = full_mnn_aff + \
#             core._mnn_affinity(mnn, mnn_scaling_factors,
#                           tp_offset[t1], tp_offset[t2], 'cpu')
        full_mnn_aff = full_mnn_aff + \
            _mnn_affinity(mnn, mnn_scaling_factors,
                          tp_offset[t1], tp_offset[t2], 'cpu')
    # Symmetrize the affinity matrix and return
    aug_aff2 = nn_aff + nn_aff.T + full_mnn_aff + full_mnn_aff.T
    aff2 = nn_aff + nn_aff.T
    return aug_aff2, aff2

'''COMPUTE RANDOM WALK PROBABILITIES'''
def random_walk_probabilities(A, labels):
    D = np.diag(np.sum(A, axis=1))
    L = D - A  # graph laplacian
    seeds = np.array([e != 0 for e in labels], dtype=bool)
    Lu = L[np.invert(seeds),:][:, np.invert(seeds)]  # unlabeled rows, unlabeled cols
    BT = L[np.invert(seeds),:][:, seeds]  # unlabeled rows, labeled cols
    classes = np.unique(labels[labels > 0])
    M = np.zeros((seeds.sum(), len(classes)))
    for k in classes:
        M[labels[seeds] == k, k-1] = 1
    P = np.linalg.lstsq(Lu, np.dot(-BT, M), rcond = None)[0]
    return P

In [4]:
# Set plotting settings
sns.set_style('white')
matplotlib.rcParams['figure.figsize'] = [4, 4]
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['image.cmap'] = 'Spectral_r'
matplotlib.rcParams['savefig.dpi'] = 150
matplotlib.style.use("ggplot")
warnings.filterwarnings(action="ignore", module="matplotlib", message="findfont")

In [4]:
# Define a sample name
sample='Metastatic_BASE_CTRL'

In [5]:
adata_org2=adata_org

In [6]:
# Load additional data
adata_patient = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/adatas/KG146_Patient_Organoid.h5ad')
adata_patient = adata_patient[~adata_patient.obs.loc[:,'Cell State'].isin(['nan','NA','SCC']),:].copy()


In [7]:
# Prepare matrices
mat_2 = pd.DataFrame(adata_org2.raw.X.todense(), index=adata_org2.raw.obs_names, columns=adata_org2.raw.var_names)
mat_1 = pd.DataFrame(adata_patient.raw.X.todense(), index=adata_patient.raw.obs_names, columns=adata_patient.raw.var_names)

mat_1.index = 'Patient_' + mat_1.index
mat_2.index = 'Org_' + mat_2.index

mat = pd.concat([mat_1, mat_2], join='outer').fillna(0).astype(int)


In [8]:
# Normalize and visualize
adata = sc.AnnData(mat)
adata.raw = adata

sc.pp.calculate_qc_metrics(adata, inplace=True)
adata.obs.loc[:,'log10GenesPerUMI'] = (np.log10(adata.obs.n_genes_by_counts)).div(np.log10(adata.obs.total_counts))

adata.obs['sample'] = [re.sub('_[0-9]+$','',i) for i in adata.obs_names]

sc.pp.filter_genes(adata, min_cells=1)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata, base=2)

adata.layers['counts'] = adata.raw[:,adata.var_names].X

adata.obs['Modality'] = adata.obs.index.str.replace('_.*','', regex=True)


In [9]:
# Feature selection with HVGs
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=500,
    layer="counts",
    flavor="seurat_v3",  # Change to "seurat"
    batch_key="Modality"
)


In [10]:
# Load cell types and markers
cell_types_df = pd.read_excel("/data/chanjlab/CRC_ZFP36L2.092023/ref/cell_type_kg.xlsx", header=None)

# Create a gene dictionary
genes_dict = {}
for index, row in cell_types_df.iterrows():
    cell_type = row.iloc[0]  # Extract the cell type
    genes = row.iloc[2:].dropna().tolist()
    genes_dict[cell_type] = genes
    
genes_list = []
for index, row in cell_types_df.iterrows():
    genes = row.iloc[2:].dropna().tolist()
    genes_list.extend(genes)

genes_list = [x for x in genes_list if x in adata.var_names]
NE_markers = ['CHGA','CHGB','SYP','ENO2','NCAM1','INSM1']
genes_to_add = list(set(genes_list + NE_markers))

# Set genes_to_add as highly variable in adata.var
adata.var.loc[genes_to_add, 'highly_variable'] = True


In [11]:
# Perform PCA
sc.tl.pca(adata, use_highly_variable=True)

# Define timepoint connections
timepoint_connections = pd.DataFrame(columns=[0, 1])
index = 0
timepoint_connections.loc[index, :] = ['Org', 'Patient']; index += 1

tp = adata.obs.Modality


In [12]:
# Compute augmented and affinity matrix
n_neighbors = 15
n_neighbors2 = 30
pca_merge = pd.DataFrame(adata.obsm['X_pca'], index=adata.obs_names)
aug_mat, aff_mat = harmony_aug_mat_with_pca(pca_merge, tp, timepoint_connections, n_neighbors, n_neighbors2)


Nearest neighbor computation...
Constucting affinities between Org and Patient...
t+1 neighbors of t...
t neighbors of t+1...


In [13]:
# Add matrices to adata
adata.obsm['aug_mat'] = aug_mat.toarray()
adata.obsm['aff_mat'] = aff_mat.toarray()

aug_mat = pd.DataFrame(adata.obsm['aug_mat'], index=adata.obs.index, columns=adata.obs.index)


In [14]:
aug_mat

,Patient_120703424285939_KG146M,Patient_120703455025013_KG146M,Patient_120718456679846_KG146M,Patient_120718468987109_KG146M,Patient_120728790953196_KG146M,Patient_120728822892973_KG146M,Patient_120769847183779_KG146M,Patient_120778570971060_KG146M,Patient_120781435485413_KG146M,Patient_120786758323435_KG146M,...,Org_KG146Li_BASE_shCtrl_GCATTAGTCGGCTGAC-1,Org_KG146Li_BASE_shCtrl_CTGTGGGTCAAGTCGT-1,Org_KG146Li_BASE_shCtrl_TATCTGTAGGCATCAG-1,Org_KG146Li_BASE_shCtrl_GTTGTGAAGCACCGAA-1,Org_KG146Li_BASE_shCtrl_GGACGTCGTCACGTGC-1,Org_KG146Li_BASE_shCtrl_AGCGATTGTAGCTGCC-1,Org_KG146Li_BASE_shCtrl_GTAGCTAAGGGACAGG-1,Org_KG146Li_BASE_shCtrl_CATAAGCTCAGTGATC-1,Org_KG146Li_BASE_shCtrl_CGTGCTTCAATTCTTC-1,Org_KG146Li_BASE_shCtrl_TCATGTTGTTCTTAGG-1
Patient_120703424285939_KG146M,2.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Patient_120703455025013_KG146M,0.0,2.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Patient_120718456679846_KG146M,0.0,0.0,4.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Patient_120718468987109_KG146M,0.0,0.0,0.0,4.00000,0.41211,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Patient_120728790953196_KG146M,0.0,0.0,0.0,0.41211,2.00000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Org_KG146Li_BASE_shCtrl_AGCGATTGTAGCTGCC-1,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,0.0
Org_KG146Li_BASE_shCtrl_GTAGCTAAGGGACAGG-1,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.413216,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0
Org_KG146Li_BASE_shCtrl_CATAAGCTCAGTGATC-1,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0
Org_KG146Li_BASE_shCtrl_CGTGCTTCAATTCTTC-1,0.0,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0,0.0


In [15]:
# Define cell states
bc_intersect = ('Patient_' + adata_patient.obs.index).intersection(adata.obs.index)
adata.obs['Cell State'] = 'nan'
adata.obs.loc[bc_intersect, 'Cell State'] = adata_patient.obs.loc[bc_intersect.str.replace('Patient_',''), 'Cell State'].values

cell_types = ['nan'] + adata.obs['Cell State'].astype('category').cat.categories[:-1].to_list()
ct_labels = adata.obs['Cell State'].astype('category').cat.reorder_categories(cell_types)
ct_codes = ct_labels.cat.codes


In [16]:
# Compute random walk probabilities
P = random_walk_probabilities(aug_mat.values, ct_codes.values)

# Create ct_prob DataFrame
ct_prob = pd.DataFrame(P, index=aug_mat.index[ct_codes==0], columns=cell_types[1:])
ct_prob.index = ct_prob.index.str.replace('Org_', '')

# Add cd_pred column
cd_pred = ct_prob.idxmax(axis=1)
ct_prob['Cell_State'] = cd_pred


In [17]:
ct_prob

,Early NET,Enterocyte-like,Fetal Progenitor,Goblet-like,ISC-like,Injury Repair,TA-like,Cell_State
KG146Li_BASE_shCtrl_GCCAGCATCAGACCGC-1,0.000652,0.134317,0.000699,0.042051,0.095548,0.011469,0.715265,TA-like
KG146Li_BASE_shCtrl_GTGAGCCCATGACTAC-1,0.001374,0.193379,0.001438,0.042653,0.154933,0.022702,0.583520,TA-like
KG146Li_BASE_shCtrl_GTGAGGAGTATCCTCC-1,0.006121,0.189442,0.007024,0.052829,0.227593,0.074269,0.442721,TA-like
KG146Li_BASE_shCtrl_AACCATGTCAGAGTTC-1,0.000291,0.065505,0.000337,0.022153,0.040345,0.004628,0.866741,TA-like
KG146Li_BASE_shCtrl_TGGCGTGAGCATTTGC-1,0.000629,0.142695,0.000618,0.030754,0.127755,0.012757,0.684792,TA-like
...,...,...,...,...,...,...,...,...
KG146Li_BASE_shCtrl_AGCGATTGTAGCTGCC-1,0.000947,0.635511,0.000410,0.011853,0.174473,0.148960,0.027846,Enterocyte-like
KG146Li_BASE_shCtrl_GTAGCTAAGGGACAGG-1,0.028310,0.259995,0.034640,0.443106,0.141749,0.081743,0.010458,Goblet-like
KG146Li_BASE_shCtrl_CATAAGCTCAGTGATC-1,0.063765,0.211183,0.214224,0.201824,0.094431,0.201084,0.013489,Fetal Progenitor
KG146Li_BASE_shCtrl_CGTGCTTCAATTCTTC-1,0.012954,0.233974,0.011505,0.658229,0.035619,0.043338,0.004382,Goblet-like


In [18]:
adata_org

AnnData object with n_obs × n_vars = 2186 × 30693
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', 'phenograph', 'leiden'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells'
    uns: 'diffmap_evals', 'leiden', 'log1p', 'neighbors', 'num_components', 'paga', 'pca', 'phenograph_sizes', 'rank_genes_groups', 'umap', 'var_explained'
    obsm: 'X_diffmap', 'X_pca', 'X_umap', 'gene_expression_encoding'
    varm: 'PCs'
    layers: 'without_log'
    obsp: 'connectivities', 'distanc

In [2]:
### I want to verify the phenograph classification. 

In [7]:
# Let's concatenate all the .csv files from phenograph classification.
# Directory containing the CSV files
directory = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_noreplicates/harmony_phenograph/'

# List all CSV files in the directory
csv_files = [file for file in os.listdir(directory) if file.endswith('.csv')]

# Initialize an empty list to store DataFrames
dfs = []

# Iterate over each CSV file and read it into a DataFrame, then append to the list
for file in csv_files:
    file_path = os.path.join(directory, file)
    df = pd.read_csv(file_path, index_col=0)
    dfs.append(df)

# Concatenate all DataFrames into a single DataFrame
concatenated_df = pd.concat(dfs)

In [26]:
concatenated_df

,Early NET,Enterocyte-like,Fetal Progenitor,Goblet-like,ISC-like,Injury Repair,TA-like,Cell_State
146Li_dedifferentiation_shZFP36L2_3_CAGTTCCCATGCCATA-1,0.083870,0.106277,0.083480,0.035359,0.176457,0.187986,0.326571,TA-like
146Li_dedifferentiation_shZFP36L2_3_TGGAGGAAGCTTCATG-1,0.051277,0.089515,0.071567,0.037297,0.133662,0.141404,0.475278,TA-like
146Li_dedifferentiation_shZFP36L2_3_CGTTGGGGTAGCGAGT-1,0.009146,0.162380,0.007881,0.137855,0.082742,0.035804,0.564192,TA-like
146Li_dedifferentiation_shZFP36L2_3_TTGCTGCTCCGCTTAC-1,0.003275,0.213747,0.002874,0.243471,0.073368,0.031337,0.431928,TA-like
146Li_dedifferentiation_shZFP36L2_3_GTAGGTTAGAATCGCG-1,0.072418,0.107836,0.135979,0.034405,0.149698,0.190878,0.308786,TA-like
...,...,...,...,...,...,...,...,...
146P_BASE_shCTRL_CGCCATTGTAATTGGA-1,0.006618,0.229360,0.010712,0.068213,0.139319,0.039362,0.506415,TA-like
146P_BASE_shCTRL_GTCTTTATCTTAGCCC-1,0.022066,0.397472,0.002688,0.017468,0.323913,0.148516,0.087878,Enterocyte-like
146P_BASE_shCTRL_AGGTCTAGTGCTTCAA-1,0.071090,0.193139,0.013350,0.103282,0.045869,0.552946,0.020324,Injury Repair
146P_BASE_shCTRL_GACACGCTCTCTGCTG-1,0.065007,0.254193,0.028845,0.190560,0.064924,0.371004,0.025468,Injury Repair


In [27]:
# Define the file path where you want to save the concatenated DataFrame
output_file_path = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_noreplicates/harmony_phenograph/adata.probabilities.full.csv'

# Save the concatenated DataFrame to a CSV file
concatenated_df.to_csv(output_file_path)

print("Concatenated DataFrame saved successfully.")

Concatenated DataFrame saved successfully.


In [38]:
# Read the imputed adata
adata_postprocessed = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_noreplicates/postprocess_adata/adata.combined.postprocess.h5ad')

In [39]:
# Iterate over each column of the concatenated DataFrame
for column in concatenated_df.columns:
    # Add the column from concatenated_df to adata_imputed.obs
    adata_postprocessed.obs[column] = concatenated_df[column].values

print("Columns added successfully.")

Columns added successfully.


In [40]:
adata_postprocessed

AnnData object with n_obs × n_vars = 29467 × 30693
    obs: 'background_fraction', 'cell_probability', 'cell_size', 'droplet_efficiency', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'log10GenesPerUMI', 'original_total_counts', 'log10_original_total_counts', 'mito_frac', 'Patient', 'Tumor_Site', 'Culture_Media', 'ZFP_Expression', 'Replicate', 'Batch', 'Sample', 'phenograph', 'leiden', 'Early NET', 'Enterocyte-like', 'Fetal Progenitor', 'Goblet-like', 'ISC-like', 'Injury Repair', 'TA-like', 'Cell_State'
    var: 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'ribo', 'n_cells'
    uns: 'diffmap_evals', 'leiden', 'log1p', 'neighbors', 'num_components', 'paga', 'phenograph_sizes', 'rank_genes_groups', 'umap', 'var_explained'
    obsm: 'X_diffmap', 'X_pca', 

In [41]:
# Save the updated AnnData object
adata_postprocessed.write_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_noreplicates/postprocess_adata/adata.combined.postprocess.probabilites.h5ad')

print("Updated AnnData object saved successfully.")

Updated AnnData object saved successfully.


In [36]:
adata_imputed.obs['Cell_State']

KG146Li_HISC_shCtrl_AGATGAAGTTCAATCG-1              TA-like
KG146Li_HISC_shCtrl_CGTTCTGCAACACAAA-1              TA-like
KG146Li_HISC_shCtrl_TGTAGACAGGAAGAAC-1              TA-like
KG146Li_HISC_shCtrl_TACCGGGTCACGGGAA-1              TA-like
KG146Li_HISC_shCtrl_CCTCTAGAGTGGATAT-1              TA-like
                                                 ...       
146P_BASE_shZFP36L2_3_GTCCTCAGTCATCACA-1            TA-like
146P_BASE_shZFP36L2_3_CCTCTAGAGTACGTCT-1    Enterocyte-like
146P_BASE_shZFP36L2_3_TGTAGACTCGGAACTT-1      Injury Repair
146P_BASE_shZFP36L2_3_CATTGTTTCTAGTGTG-1      Injury Repair
146P_BASE_shZFP36L2_3_CAGCAATAGTTGGAAT-1        Goblet-like
Name: Cell_State, Length: 29467, dtype: category
Categories (7, object): ['Early NET', 'Enterocyte-like', 'Fetal Progenitor', 'Goblet-like', 'ISC-like', 'Injury Repair', 'TA-like']

In [35]:
sc.pl.umap(adata_imputed, 
          color='ISC-like')

In [21]:
# Filter the DataFrame to include only rows where 'Cell_State' is 'Early_NET'
early_net_cells = concatenated_df[concatenated_df['Cell_State'] == 'Early NET']

In [20]:
early_net_cells

,Early NET,Enterocyte-like,Fetal Progenitor,Goblet-like,ISC-like,Injury Repair,TA-like,Cell_State
146Li_dedifferentiation_shZFP36L2_3_ATTTCACAGCACACAG-1,0.267386,0.103866,0.153087,0.041961,0.119034,0.214226,0.100439,Early NET
146Li_dedifferentiation_shZFP36L2_3_AACCTGACATGAAGCG-1,0.264758,0.133478,0.100542,0.070258,0.145970,0.170630,0.114365,Early NET
146Li_dedifferentiation_shZFP36L2_3_CAACCTCGTCAGTCTA-1,0.345817,0.073248,0.190741,0.025409,0.087235,0.215700,0.061851,Early NET
146Li_dedifferentiation_shZFP36L2_3_TGTGATGGTTCGCGTG-1,0.202599,0.139736,0.112568,0.074979,0.153715,0.175981,0.140422,Early NET
146Li_dedifferentiation_shZFP36L2_3_GTGACGCAGATACTGA-1,0.364847,0.079883,0.173055,0.029922,0.096663,0.202485,0.053145,Early NET
...,...,...,...,...,...,...,...,...
146P_BASE_shCTRL_GACTATGAGGAGAGGC-1,0.592728,0.054131,0.061556,0.013619,0.032410,0.241119,0.004438,Early NET
146P_BASE_shCTRL_CCTTTGGTCTAAGAAG-1,0.381772,0.196265,0.050616,0.121823,0.068550,0.166989,0.013985,Early NET
146P_BASE_shCTRL_ACCTACCCAGCGCGTT-1,0.559756,0.048098,0.083436,0.035370,0.034032,0.224730,0.014578,Early NET
146P_BASE_shCTRL_CTCAGAAGTATGCTTG-1,0.559737,0.052884,0.058738,0.006797,0.029743,0.290521,0.001580,Early NET


In [13]:
concatenated_df.columns

Index(['Early NET', 'Enterocyte-like', 'Fetal Progenitor', 'Goblet-like',
       'ISC-like', 'Injury Repair', 'TA-like', 'Cell_State'],
      dtype='object')

In [24]:
# Find the indices
indices = early_net_cells[early_net_cells.index.str.contains("P") & 
                          early_net_cells.index.str.contains("BASE") & 
                          early_net_cells.index.str.contains("CTRL")].index

# Extract the rows based on the indices
filtered_rows = early_net_cells.loc[indices]


In [25]:
filtered_rows

,Early NET,Enterocyte-like,Fetal Progenitor,Goblet-like,ISC-like,Injury Repair,TA-like,Cell_State
146P_BASE_shCTRL_GGAATGGGTATCCTTT-1,0.587028,0.047315,0.141943,0.066021,0.033856,0.116030,0.007806,Early NET
146P_BASE_shCTRL_TAATTCCCAACGCCCA-1,0.407418,0.113238,0.050348,0.164998,0.084378,0.159115,0.020504,Early NET
146P_BASE_shCTRL_GAGTCTACAACACGTT-1,0.544815,0.106505,0.059086,0.057114,0.040637,0.184038,0.007805,Early NET
146P_BASE_shCTRL_TCCACCATCACGTCCT-1,0.571289,0.056674,0.057411,0.009312,0.029645,0.273630,0.002038,Early NET
146P_BASE_shCTRL_TGCGACGAGGCTAGCA-1,0.436322,0.052174,0.215775,0.046538,0.039520,0.188225,0.021447,Early NET
146P_BASE_shCTRL_TTTGGAGTCTCCGAGG-1,0.315151,0.118343,0.043165,0.183490,0.146689,0.172016,0.021147,Early NET
146P_BASE_shCTRL_AACTTCTGTTCCTAGA-1,0.514921,0.091481,0.057444,0.044260,0.036250,0.250006,0.005638,Early NET
146P_BASE_shCTRL_TCTATACCAATAGGAT-1,0.228875,0.208747,0.035096,0.222950,0.120879,0.151639,0.031814,Early NET
146P_BASE_shCTRL_TCACAAGCAATTGGTC-1,0.258415,0.147196,0.048994,0.219695,0.105844,0.195522,0.024334,Early NET
146P_BASE_shCTRL_CATTCCGGTATCTTCT-1,0.406756,0.063713,0.077471,0.066336,0.031672,0.327352,0.026700,Early NET
